# 13-тарау. Keras-пен терең оқыту

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 13.1. Неге кітапхана керек?

## 13.2. MNIST деректер жиыны

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(42)       # нәтижелерді қайталау үшін

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("Оқыту:", x_train.shape, y_train.shape)
print("Тест: ", x_test.shape, y_test.shape)
print("Пиксель мәндері:", x_train.min(), "...", x_train.max())

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

## 13.3. Модельді құрастыру

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),               # кіріс: 28×28 сурет
    layers.Flatten(),                          # 28×28 → 784 вектор
    layers.Dense(128, activation="relu"),      # жасырын қабат
    layers.Dense(10, activation="softmax"),    # шығыс: 10 цифр
])
model.summary()

## 13.4. compile: оқытуға дайындық

In [ ]:
model.compile(
    optimizer="adam",                            # оңтайландыру алгоритмі
    loss="sparse_categorical_crossentropy",      # қате функциясы
    metrics=["accuracy"],                        # бақылайтын көрсеткіш
)

## 13.5. fit: оқыту

In [ ]:
history = model.fit(x_train, y_train, epochs=10, batch_size=64,
                    validation_split=0.1, verbose=0)

h = history.history                      # әр эпохадағы көрсеткіштер
for e in range(10):
    print(f"Эпоха {e + 1:2d}: оқыту = {h['accuracy'][e]:.4f}, "
          f"валидация = {h['val_accuracy'][e]:.4f}")

## 13.6. evaluate және predict

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Тест дәлдігі: {test_acc:.2%}")

probs = model.predict(x_test[:8], verbose=0)     # (8, 10) ықтималдықтар
print("Болжамдар:  ", probs.argmax(axis=1))
print("Дұрыс жауап:", y_test[:8])

## 13.7. Асыра оқытуды байқау

In [ ]:
def build_model(dropout_rate=0.0):
    keras.utils.set_random_seed(42)
    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(512, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

results = {}
for rate in [0.0, 0.3]:
    m = build_model(rate)
    hist = m.fit(x_train, y_train, epochs=20, batch_size=128,
                 validation_split=0.1, verbose=0).history
    results[rate] = hist
    best = int(np.argmin(hist["val_loss"]))
    print(f"Dropout {rate}: ең аз val_loss {hist['val_loss'][best]:.4f} "
          f"({best + 1}-эпоха), соңғы {hist['val_loss'][-1]:.4f}, "
          f"val_acc {hist['val_accuracy'][-1]:.4f}")

## 13.8. Ерте тоқтату

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",          # нені бақылаймыз
    patience=3,                  # неше эпоха шыдаймыз
    restore_best_weights=True,   # ең жақсы салмақтарға оралу
)
m = build_model(0.0)
hist = m.fit(x_train, y_train, epochs=50, batch_size=128,
             validation_split=0.1, callbacks=[early_stop], verbose=0)
print("Оқыту тоқтаған эпоха:", len(hist.history["loss"]))
print(f"Тест дәлдігі: {m.evaluate(x_test, y_test, verbose=0)[1]:.4f}")

## 13.9. Модельді сақтау

In [ ]:
model.save("mnist_model.keras")
loaded = keras.models.load_model("mnist_model.keras")
print(loaded.predict(x_test[:1], verbose=0).argmax())